# FASHN VTON v1.5 on a free Colab GPU
Virtual try-on, Apache-2.0 (commercial use allowed). Set **Runtime → Change runtime type → T4 GPU** first.

*Written from the official README; not yet run end to end. If a cell errors, paste the message back.*

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Install (about 2-4 min)
!git clone -q https://github.com/fashn-AI/fashn-vton-1.5.git
%cd /content/fashn-vton-1.5
!pip install -q .
# Colab's CUDA often doesn't match onnxruntime-gpu; the pose model is small, so CPU is fine.
!pip uninstall -y -q onnxruntime-gpu && pip install -q onnxruntime
# Make sure the running kernel can see the package (fixes ModuleNotFoundError: fashn_vton)
import sys, importlib
sys.path.insert(0, "/content/fashn-vton-1.5/src")
importlib.invalidate_caches()
import fashn_vton; print("fashn_vton OK:", fashn_vton.__file__)

In [ ]:
# 2. Download weights (~2 GB) - the human parser (~244 MB) downloads on first run
%cd /content/fashn-vton-1.5
!python scripts/download_weights.py --weights-dir ./weights

In [ ]:
# 3. Load the model. A T4 has no bf16, so it runs in float32 (slower, but should fit in 15 GB).
import time, torch
from fashn_vton import TryOnPipeline
t=time.time()
pipeline = TryOnPipeline(weights_dir="./weights")
print("loaded in", round(time.time()-t,1), "s | GPU:", torch.cuda.is_available())

In [ ]:
# 4. Upload your photos: first the PERSON (full body), then the GARMENT (clear photo)
from google.colab import files
from PIL import Image
import io
print("Upload the PERSON photo"); p=files.upload(); person=Image.open(io.BytesIO(next(iter(p.values())))).convert("RGB")
print("Upload the GARMENT photo"); g=files.upload(); garment=Image.open(io.BytesIO(next(iter(g.values())))).convert("RGB")
display(person.resize((256, int(256*person.height/person.width))), garment.resize((256, int(256*garment.height/garment.width))))

In [ ]:
# 5. Run. category: "tops" | "bottoms" | "one-pieces"  (use one-pieces for saree / lehenga / dress / kurta set to test)
CATEGORY = "tops"
t=time.time()
result = pipeline(person_image=person, garment_image=garment, category=CATEGORY)
print("took", round(time.time()-t,1), "s")
out = result.images[0]
out.save("output.png"); display(out)
files.download("output.png")

## Test plan
Try each of these and note what breaks: shirt, kurta (`tops`), kurta + pyjama as `one-pieces`, saree, lehenga, a person holding an object (the tablet failure), sitting pose, a dark garment on a light one.
Photos of real customers: only use ones you have consent for.